# CS 521 HW4, Problem 2: Certified Training for Multiple Norms (CURE)

Open this notebook in Google Colab and select a GPU runtime (*Runtime > Change runtime type > T4 GPU*).

Complete each part that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the verifier, the evaluation or the default hyperparameters.
Change them only if the homework tells you to.

**Threat model.** Let $f$ be the network and $x$ an image with the label $y$. The network is *certified* at $x$
for the norm $p$ if the verifier proves this: $f_y(x') - f_j(x') > 0$ for all $j \neq y$ and all $x'$ with
$\|x' - x\|_p \leq \epsilon_p$. For $\ell_\infty$, $x'$ must also be in $[0, 1]^{784}$.
We use $\epsilon_\infty = 0.1$ and $\epsilon_2 = 1.0$.
The *union certified accuracy* is the percentage of test images where the network is certified for the two norms.

In [1]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

EPS = {"linf": 0.1, "l2": 1.0}     # the radii of the threat model
NORMS = ["linf", "l2"]

device: cuda


# Data and model

The data and the network are the same as in Problem 1. The network has 3 hidden layers with 50 neurons.
It has no normalization layer.

In [2]:
def load_mnist(train):
    ds = torchvision.datasets.MNIST("data", train=train, download=True)
    x = ds.data.float().div(255).unsqueeze(1)  # (n, 1, 28, 28)
    return x.to(device), ds.targets.to(device)


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for i in range(0, len(x), batch_size):
        j = idx[i:i + batch_size]
        yield x[j], y[j]


x_train, y_train = load_mnist(train=True)
x_test, y_test = load_mnist(train=False)
print(x_train.shape, x_test.shape)


def make_model():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 50), nn.ReLU(),
        nn.Linear(50, 50), nn.ReLU(),
        nn.Linear(50, 50), nn.ReLU(),
        nn.Linear(50, 10),
    ).to(device)


def linear_layers(model):
    return [m for m in model if isinstance(m, nn.Linear)]

100%|██████████| 9.91M/9.91M [00:00<00:00, 20.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 494kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.58MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.6MB/s]


torch.Size([60000, 1, 28, 28]) torch.Size([10000, 1, 28, 28])


# Attacks

`pgd(model, x, y, eps, norm, steps)` is an untargeted PGD attack with a random start.
Use `norm="linf"` or `norm="l2"`. The function returns the adversarial images of the restart with the highest loss.

In [3]:
def _project(delta, eps, norm):
    if norm == "linf":
        return delta.clamp(-eps, eps)
    n = delta.flatten(1).norm(dim=1).clamp(min=1e-12).view(-1, 1, 1, 1)
    return delta * (eps / n).clamp(max=1.0)


def pgd(model, x, y, eps, norm, steps=10, alpha=None, restarts=1):
    alpha = alpha or 2.5 * eps / steps
    best_x, best_loss = x.clone(), torch.full((len(x),), -float("inf"), device=x.device)
    for _ in range(restarts):
        if norm == "linf":
            delta = (torch.rand_like(x) * 2 - 1) * eps
        else:
            delta = torch.randn_like(x)
            r = torch.rand(len(x), 1, 1, 1, device=x.device) * eps
            delta = delta / delta.flatten(1).norm(dim=1).view(-1, 1, 1, 1) * r
        delta = (x + delta).clamp(0, 1) - x
        for _ in range(steps):
            delta.requires_grad_(True)
            loss = F.cross_entropy(model(x + delta), y)
            grad, = torch.autograd.grad(loss, delta)
            with torch.no_grad():
                if norm == "linf":
                    step = grad.sign()
                else:
                    step = grad / grad.flatten(1).norm(dim=1).clamp(min=1e-12).view(-1, 1, 1, 1)
                delta = _project(delta + alpha * step, eps, norm)
                delta = (x + delta).clamp(0, 1) - x
        with torch.no_grad():
            loss = F.cross_entropy(model(x + delta), y, reduction="none")
            better = loss > best_loss
            best_x[better], best_loss[better] = (x + delta)[better], loss[better]
    return best_x.detach()

# Verifier

A *region* has a center `c` (a batch of images), a radius `r` and a norm.
For $\ell_\infty$, the region is the box $[c - r, c + r] \cap [0, 1]^{784}$.
For $\ell_2$, the region is the ball $\{x' : \|x' - c\|_2 \leq r\}$. We do not intersect the ball with $[0, 1]^{784}$.
This is sound, but the bounds are a little looser.

* `ibp_margin_bounds(model, c, r, norm, y)` is IBP from Problem 1. For $\ell_2$, the bounds of the first linear layer
  are exact: $W_i c + b_i \pm r \|W_i\|_2$ for each neuron $i$. The last linear layer is merged with the specification.
* `crown_margin_bounds(model, c, r, norm, y)` is CROWN. At the input, $\min_{x'} a^\top x'$ over the $\ell_2$ ball is
  $a^\top c - r \|a\|_2$.
* `certified(model, x, y, norm)` uses the larger of the two lower bounds.

The two bound functions return a tensor `m` with the shape `(batch, 10)`.
The value `m[:, j]` is a lower bound of $f_y - f_j$ on the region, and `m[:, y] = 0`. Do not change the verifier.

In [4]:
def first_layer_bounds(layer, c, r, norm):
    c = c.flatten(1)
    if norm == "linf":
        l, u = (c - r).clamp(0, 1), (c + r).clamp(0, 1)
        mid, rad = layer((u + l) / 2), ((u - l) / 2) @ layer.weight.abs().t()
    else:
        mid, rad = layer(c), r * layer.weight.norm(dim=1)
    return mid - rad, mid + rad


def spec_matrix(y, n_classes=10):
    eye = torch.eye(n_classes, device=y.device)
    return eye[y].unsqueeze(1) - eye.unsqueeze(0)  # (batch, 10, 10): row j is e_y - e_j


def ibp_margin_bounds(model, c, r, norm, y):
    layers = linear_layers(model)
    l, u = first_layer_bounds(layers[0], c, r, norm)
    for layer in layers[1:-1]:
        l, u = F.relu(l), F.relu(u)
        mid, rad = layer((u + l) / 2), ((u - l) / 2) @ layer.weight.abs().t()
        l, u = mid - rad, mid + rad
    l, u = F.relu(l), F.relu(u)
    last = layers[-1]
    w = last.weight[y].unsqueeze(1) - last.weight.unsqueeze(0)  # (batch, 10, n): W_y - W_j
    b = last.bias[y].unsqueeze(1) - last.bias.unsqueeze(0)
    mid, rad = (u + l) / 2, (u - l) / 2
    return (w @ mid.unsqueeze(2)).squeeze(2) - (w.abs() @ rad.unsqueeze(2)).squeeze(2) + b


def _relu_relaxation(l, u):
    active, unstable = l >= 0, (l < 0) & (u > 0)
    s = torch.where(active, torch.ones_like(l), torch.zeros_like(l))
    s = torch.where(unstable, u / (u - l).clamp(min=1e-12), s)
    c = torch.where(unstable, -s * l, torch.zeros_like(l))
    alpha = torch.where(active | (unstable & (u > -l)), torch.ones_like(l), torch.zeros_like(l))
    return alpha, s, c


def _backward_lower(A, layers, bounds, c0, r, norm):
    # Lower bound of A z, where z is the output of layers[-1] and bounds[i] are the bounds of the output of layers[i].
    const = torch.zeros(A.shape[:2], device=A.device)
    for i in range(len(layers) - 1, -1, -1):
        const = const + A @ layers[i].bias
        A = A @ layers[i].weight
        if i > 0:
            alpha, s, c = _relu_relaxation(*bounds[i - 1])
            a_pos, a_neg = A.clamp(min=0), A.clamp(max=0)
            const = const + (a_neg * c.unsqueeze(1)).sum(2)
            A = a_pos * alpha.unsqueeze(1) + a_neg * s.unsqueeze(1)
    c0 = c0.flatten(1)
    if norm == "linf":
        l0, u0 = (c0 - r).clamp(0, 1), (c0 + r).clamp(0, 1)
        return const + (A.clamp(min=0) * l0.unsqueeze(1)).sum(2) + (A.clamp(max=0) * u0.unsqueeze(1)).sum(2)
    return const + (A @ c0.unsqueeze(2)).squeeze(2) - r * A.norm(dim=2)


def crown_margin_bounds(model, c, r, norm, y):
    layers = linear_layers(model)
    bounds = []
    for k in range(len(layers) - 1):
        eye = torch.eye(layers[k].out_features, device=c.device).expand(len(c), -1, -1)
        lo = _backward_lower(eye, layers[:k + 1], bounds, c, r, norm)
        hi = -_backward_lower(-eye, layers[:k + 1], bounds, c, r, norm)
        bounds.append((lo, hi))
    return _backward_lower(spec_matrix(y), layers, bounds, c, r, norm)


@torch.no_grad()
def certified(model, x, y, norm, eps=None):
    eps = EPS[norm] if eps is None else eps
    m = torch.maximum(ibp_margin_bounds(model, x, eps, norm, y), crown_margin_bounds(model, x, eps, norm, y))
    m[torch.arange(len(y)), y] = float("inf")
    return m.min(dim=1).values > 0

# Evaluation

`evaluate(model, name)` uses the 10,000 test images. It returns the clean accuracy, the certified accuracy for each
norm and the union certified accuracy. With `attacks=True`, it also returns the PGD accuracy (50 steps) for each norm
and for the union. The PGD accuracy is an upper bound of the true robust accuracy.
Use `evaluate` for all the numbers in your report.

In [5]:
def evaluate(model, name, attacks=False, n=10000, batch_size=1000):
    model.eval()
    counts = {k: 0 for k in ["clean", "linf", "l2", "union", "pgd_linf", "pgd_l2", "pgd_union"]}
    for x, y in batches(x_test[:n], y_test[:n], batch_size, False):
        with torch.no_grad():
            correct = model(x).argmax(1) == y
        cert = {p: certified(model, x, y, p) & correct for p in NORMS}
        counts["clean"] += correct.sum().item()
        counts["linf"] += cert["linf"].sum().item()
        counts["l2"] += cert["l2"].sum().item()
        counts["union"] += (cert["linf"] & cert["l2"]).sum().item()
        if attacks:
            rob = {}
            for p in NORMS:
                x_adv = pgd(model, x, y, EPS[p], p, steps=50)
                with torch.no_grad():
                    rob[p] = correct & (model(x_adv).argmax(1) == y)
                counts["pgd_" + p] += rob[p].sum().item()
            counts["pgd_union"] += (rob["linf"] & rob["l2"]).sum().item()
    res = {"name": name}
    res.update({k: 100 * v / n for k, v in counts.items() if attacks or not k.startswith("pgd")})
    return res


def print_results(rows):
    for r in rows:
        print(f"{r['name']:<26}" + "  ".join(f"{k} {v:6.2f}" for k, v in r.items()
                                     if k != "time" and isinstance(v, float)) +
              (f"  time/epoch {r['time']:5.1f} s" if "time" in r else ""))

# Training loop

For each batch, `train(model, loss_fn)` calls `loss_fn(model, x, y, t)` and does one optimizer step.
The value `t` is the training progress in epochs (a float). The schedule is the same as in Problem 1:

* For $t < 1$, `eps_scale(t)` is 0 (standard training).
* For $1 \leq t < 11$, `eps_scale(t)` increases linearly from 0 to 1.
* For $t \geq 11$, `eps_scale(t)` is 1.

A loss function uses the radius `eps_scale(t) * EPS[norm]`.

If you give the argument `gp_beta`, `train` uses `gradient_projection` (part (b)) when the radius is at its
final value ($t \geq 11$).

In [6]:
WARMUP, RAMP = 1, 10
LR = 4e-3


def eps_scale(t):
    return 0.0 if t <= WARMUP else min((t - WARMUP) / RAMP, 1.0)


def train(model, loss_fn, epochs=20, lr=LR, batch_size=512, seed=0, gp_beta=None, verbose=True):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[15], gamma=0.1)
    params = [p for p in model.parameters() if p.requires_grad]
    n_batches = (len(x_train) + batch_size - 1) // batch_size
    start = time.time()
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for i, (x, y) in enumerate(batches(x_train, y_train, batch_size, True)):
            t = epoch + i / n_batches
            loss = loss_fn(model, x, y, t)
            opt.zero_grad()
            if gp_beta is not None and t >= WARMUP + RAMP:
                g_c = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, gn, gc in zip(params, g_n, g_c):
                    p.grad = gradient_projection(gn, gc, gp_beta)
            else:
                loss.backward()
            opt.step()
            total += loss.item()
        sched.step()
        if verbose:
            print(f"epoch {epoch + 1:2d}  loss {total / n_batches:.4f}  time {time.time() - start:.0f} s")
    return (time.time() - start) / epochs


def standard_loss(model, x, y, t):
    return F.cross_entropy(model(x), y)

# (a) Single-norm certified training

CURE uses the SABR loss from the lectures for each norm. SABR finds an adversarial example $x^*$ with PGD, and then it
calculates the IBP loss on a small ball of radius $\tau = \lambda \epsilon$ around $x^*$. PGD searches for $x^*$ in the
ball of radius $\epsilon - \tau$ around $x$, so that the small ball is inside the full ball.

1. `propagation_region`: Find a center $x^*$ with `pgd` (5 steps) in the ball of radius $\epsilon - \tau$ around $x$.
   Attack the model in eval mode. Then set the model back to train mode.
2. `certified_loss`: Calculate the cross-entropy of $\hat{z} = -\underline{m}$. Here, $\underline{m}$ are the IBP lower
   bounds of the margins on the small ball. Thus $\hat{z}_y = 0$.
   With `reduction="none"`, return the loss of each image.
3. `single_norm_loss`: For $t < 1$, return the standard loss. After that, return `certified_loss` with the radius
   `eps_scale(t) * EPS[norm]`.

The default ratios are `LAM = {"linf": 0.7, "l2": 0.7}`.

In [7]:
LAM = {"linf": 0.7, "l2": 0.7}


def propagation_region(model, x, y, eps, lam, norm):
    # Returns (center, tau).
    # TODO: tau = lam * eps; PGD center in the (eps - tau)-ball around x
    tau = lam * eps
    search = eps - tau
    if search <=0:
      return x.detach(), tau
    model.eval()
    center = pgd(model, x, y, search, norm, steps=5)
    model.train()
    return center, tau


def certified_loss(model, x, y, eps, lam, norm, reduction="mean"):
    # TODO: CE of the negative IBP margin bounds on the small region
    center, tau = propagation_region(model, x, y, eps, lam, norm)
    zhat = -ibp_margin_bounds(model, center, tau, norm, y)
    return F.cross_entropy(zhat, y, reduction=reduction)


def single_norm_loss(model, x, y, t, norm):
    # TODO: standard loss for t < 1, then the certified loss with the scheduled radius
    if t < WARMUP:
        return standard_loss(model, x, y, t)
    eps = eps_scale(t) * EPS[norm]
    return certified_loss(model, x, y, eps, LAM[norm], norm)

In [8]:
models, results = {}, []
for name, loss_fn in [("standard", standard_loss),
                      ("linf", lambda m, x, y, t: single_norm_loss(m, x, y, t, "linf")),
                      ("l2", lambda m, x, y, t: single_norm_loss(m, x, y, t, "l2"))]:
    models[name] = make_model()
    sec = train(models[name], loss_fn)
    results.append(dict(evaluate(models[name], name, attacks=True), time=sec))
print_results(results)

epoch  1  loss 0.5739  time 1 s
epoch  2  loss 0.2066  time 1 s
epoch  3  loss 0.1526  time 1 s
epoch  4  loss 0.1249  time 1 s
epoch  5  loss 0.1037  time 1 s
epoch  6  loss 0.0921  time 2 s
epoch  7  loss 0.0788  time 2 s
epoch  8  loss 0.0659  time 2 s
epoch  9  loss 0.0631  time 2 s
epoch 10  loss 0.0522  time 2 s
epoch 11  loss 0.0483  time 3 s
epoch 12  loss 0.0440  time 3 s
epoch 13  loss 0.0357  time 3 s
epoch 14  loss 0.0333  time 3 s
epoch 15  loss 0.0381  time 4 s
epoch 16  loss 0.0188  time 4 s
epoch 17  loss 0.0139  time 4 s
epoch 18  loss 0.0122  time 4 s
epoch 19  loss 0.0112  time 4 s
epoch 20  loss 0.0105  time 5 s
epoch  1  loss 0.5883  time 0 s
epoch  2  loss 0.4867  time 6 s
epoch  3  loss 0.4756  time 8 s
epoch  4  loss 0.4792  time 9 s
epoch  5  loss 0.4964  time 10 s
epoch  6  loss 0.5258  time 12 s
epoch  7  loss 0.5676  time 13 s
epoch  8  loss 0.6234  time 15 s
epoch  9  loss 0.6936  time 17 s
epoch 10  loss 0.7626  time 19 s
epoch 11  loss 0.8395  time 20 s
e

# (b) CURE

Let $\hat{z}^\infty$ and $\hat{z}^2$ be the negative IBP margin bounds on the two small balls of an image.
These are the same values as in `certified_loss`. Let $d^p = \mathrm{softmax}(\hat{z}^p)$. Complete `cure_loss`:
$$\mathcal{L}_{CURE} = \frac{1}{n} \sum_{i=1}^{n} \max\left(\mathcal{L}_{CE}(\hat{z}^\infty_i, y_i),\ \mathcal{L}_{CE}(\hat{z}^2_i, y_i)\right)
+ \eta \cdot \frac{1}{|\gamma|} \sum_{i \in \gamma} \mathrm{KL}\left(d^2_i \,\|\, d^\infty_i\right),$$
Here, $\gamma$ is the set of images in the batch that are certified for $\ell_2$ on their small ball.
For these images, all entries of $\hat{z}^2_i$ except $\hat{z}^2_{i, y_i}$ are negative.
The first term is CURE-Max. The second term is the bound alignment (BA).
For $t < 1$, return the standard loss. If $\gamma$ is empty, the second term is 0.

Then complete `gradient_projection` (GP). Let $g_c$ be the gradient of `cure_loss`, and $g_n$ the gradient of the
cross-entropy loss on the clean images. For each parameter tensor:
$$g = \beta \cdot \mathrm{GP}(g_n, g_c) + (1 - \beta) \cdot g_c, \qquad
\mathrm{GP}(g_n, g_c) = \begin{cases} \cos(g_n, g_c) \cdot g_n & \text{if } \cos(g_n, g_c) > 0 \\ 0 & \text{otherwise.} \end{cases}$$
This is the same operation as in RAMP.

This is a simplified version of CURE. The paper is different in three ways:

* It propagates an $\ell_\infty$ box also for $\ell_2$.
* It uses a different $\ell_2$ attack.
* It applies GP to the updates of a full epoch.

In [9]:
def cure_loss(model, x, y, t, eta=0.5):
    if t < WARMUP:
        return standard_loss(model, x, y, t)
    # TODO: per-image max of the two certified losses + eta * KL on the l2-certified subset
    zhat = {}
    for p in NORMS:
        center, tau = propagation_region(model, x, y, eps_scale(t) * EPS[p], LAM[p], p)
        zhat[p] = -ibp_margin_bounds(model, center, tau, p, y)  # same zhat as in certified_loss

    # per-image max of the two SABR losses, then mean over the batch
    ce = torch.stack([F.cross_entropy(zhat[p], y, reduction="none") for p in NORMS])  # (2, batch)
    loss = ce.max(dim=0).values.mean()

    # eta * KL on the l2-certified subset
    if eta > 0:
        z2 = zhat["l2"].detach().clone()
        z2[torch.arange(len(y)), y] = -float("inf")
        gamma = z2.max(dim=1).values < 0  # all entries except zhat_y are negative
        if gamma.any():
            log_d2 = F.log_softmax(zhat["l2"][gamma], dim=1)
            log_dinf = F.log_softmax(zhat["linf"][gamma], dim=1)
            kl = F.kl_div(log_dinf, log_d2, log_target=True, reduction="batchmean")
            loss = loss + eta * kl
    return loss

def gradient_projection(g_n, g_c, beta):
    # TODO: beta * GP(g_n, g_c) + (1 - beta) * g_c for one parameter tensor
    cos = (g_n * g_c).sum() / (g_n.norm() * g_c.norm()).clamp(min=1e-12)
    gp = cos * g_n if cos > 0 else torch.zeros_like(g_n)
    return beta * gp + (1 - beta) * g_c

In [10]:
results = results[:3]  # keep only part a's rows
for name, loss_fn, beta in [("cure_max", lambda m, x, y, t: cure_loss(m, x, y, t, eta=0.0), None),
                            ("cure_max_ba", cure_loss, None),
                            ("cure_max_ba_gp", cure_loss, 0.5)]:
    runs = []
    for seed in [0, 1]:
        torch.manual_seed(seed)
        models[name] = make_model()
        sec = train(models[name], loss_fn, seed=seed, gp_beta=beta)
        runs.append(dict(evaluate(models[name], name, attacks=True), time=sec))
    results.append({k: (runs[0][k] + runs[1][k]) / 2 if k != "name" else name for k in runs[0]})
print_results(results)

epoch  1  loss 0.5883  time 0 s
epoch  2  loss 0.4887  time 3 s
epoch  3  loss 0.4981  time 6 s
epoch  4  loss 0.5257  time 9 s
epoch  5  loss 0.5670  time 11 s
epoch  6  loss 0.6305  time 14 s
epoch  7  loss 0.7240  time 17 s
epoch  8  loss 0.8318  time 20 s
epoch  9  loss 0.9499  time 22 s
epoch 10  loss 1.0798  time 25 s
epoch 11  loss 1.2188  time 28 s
epoch 12  loss 1.2717  time 31 s
epoch 13  loss 1.2410  time 33 s
epoch 14  loss 1.2204  time 36 s
epoch 15  loss 1.2015  time 38 s
epoch 16  loss 1.1524  time 41 s
epoch 17  loss 1.1397  time 44 s
epoch 18  loss 1.1343  time 47 s
epoch 19  loss 1.1298  time 49 s
epoch 20  loss 1.1272  time 52 s
epoch  1  loss 0.5640  time 0 s
epoch  2  loss 0.4954  time 3 s
epoch  3  loss 0.5022  time 6 s
epoch  4  loss 0.5396  time 8 s
epoch  5  loss 0.5988  time 11 s
epoch  6  loss 0.6835  time 14 s
epoch  7  loss 0.7776  time 16 s
epoch  8  loss 0.8806  time 19 s
epoch  9  loss 0.9901  time 22 s
epoch 10  loss 1.1129  time 24 s
epoch 11  loss 1.2

# (c) Your method

Write the loss of your method in `my_loss`. Train it with `train` and the default settings, as for the other methods.
If your method needs a different update, write your own copy of `train`.
Do not change the verifier or `evaluate`.

In [11]:
def crown_ibp_margin_bounds(model, c, r, norm, y):
    # IBP intermediate bounds, then one CROWN backward pass from the margin spec
    layers = linear_layers(model)
    l, u = first_layer_bounds(layers[0], c, r, norm)
    bounds = [(l, u)]
    for layer in layers[1:-1]:
        lr, ur = F.relu(l), F.relu(u)
        mid, rad = layer((ur + lr) / 2), ((ur - lr) / 2) @ layer.weight.abs().t()
        l, u = mid - rad, mid + rad
        bounds.append((l, u))
    return _backward_lower(spec_matrix(y), layers, bounds, c, r, norm)


def my_loss(model, x, y, t, crown_norm='l2'):
    if t < WARMUP:
        return standard_loss(model, x, y, t)
    ce = []
    for p in NORMS:
        center, tau = propagation_region(model, x, y, eps_scale(t) * EPS[p], LAM[p], p)
        m = ibp_margin_bounds(model, center, tau, p, y)
        if p == crown_norm:
            m = crown_ibp_margin_bounds(model, center, tau, p, y)
        ce.append(F.cross_entropy(-m, y, reduction="none"))  # zhat_y = 0, as in certified_loss
    return torch.stack(ce).max(dim=0).values.mean()

In [12]:
def run_two_seeds(name, loss_fn, epochs=20, gp_beta=None):
    runs = []
    for seed in [0, 1]:
        torch.manual_seed(seed)
        models[name] = make_model()
        sec = train(models[name], loss_fn, epochs=epochs, seed=seed, gp_beta=gp_beta)
        runs.append(dict(evaluate(models[name], name, attacks=True), time=sec))
    return {k: (runs[0][k] + runs[1][k]) / 2 if k != "name" else name for k in runs[0]}


cure_row = next(r for r in results if r["name"] == "cure_max")
mine_row = run_two_seeds("cure_max_crown_l2", my_loss)
results_c = [cure_row, mine_row]
print_results(results_c)

epoch  1  loss 0.5883  time 0 s
epoch  2  loss 0.4866  time 4 s
epoch  3  loss 0.4736  time 7 s
epoch  4  loss 0.4856  time 10 s
epoch  5  loss 0.5295  time 14 s
epoch  6  loss 0.5937  time 17 s
epoch  7  loss 0.6725  time 21 s
epoch  8  loss 0.7731  time 24 s
epoch  9  loss 0.8819  time 27 s
epoch 10  loss 1.0073  time 30 s
epoch 11  loss 1.1445  time 33 s
epoch 12  loss 1.2033  time 37 s
epoch 13  loss 1.1746  time 40 s
epoch 14  loss 1.1614  time 43 s
epoch 15  loss 1.1555  time 46 s
epoch 16  loss 1.1164  time 49 s
epoch 17  loss 1.1082  time 52 s
epoch 18  loss 1.1047  time 56 s
epoch 19  loss 1.1024  time 59 s
epoch 20  loss 1.1004  time 62 s
epoch  1  loss 0.5640  time 0 s
epoch  2  loss 0.4938  time 3 s
epoch  3  loss 0.4800  time 7 s
epoch  4  loss 0.4975  time 10 s
epoch  5  loss 0.5505  time 13 s
epoch  6  loss 0.6345  time 16 s
epoch  7  loss 0.7165  time 20 s
epoch  8  loss 0.8152  time 23 s
epoch  9  loss 0.9258  time 26 s
epoch 10  loss 1.0438  time 30 s
epoch 11  loss 1

In [13]:
ratio = mine_row["time"] / cure_row["time"]
print(f"time ratio (mine / CURE-Max): {ratio:.2f}")
if ratio > 1.05:
    epochs_matched = round(20 * ratio)
    cure_long_row = run_two_seeds(f"cure_max_{epochs_matched}ep",
                                  lambda m, x, y, t: cure_loss(m, x, y, t, eta=0.0), epochs=epochs_matched)
    results_c.append(cure_long_row)
print_results(results_c)

time ratio (mine / CURE-Max): 1.20
epoch  1  loss 0.5883  time 0 s
epoch  2  loss 0.4887  time 3 s
epoch  3  loss 0.4981  time 6 s
epoch  4  loss 0.5257  time 8 s
epoch  5  loss 0.5670  time 11 s
epoch  6  loss 0.6305  time 13 s
epoch  7  loss 0.7240  time 16 s
epoch  8  loss 0.8318  time 19 s
epoch  9  loss 0.9499  time 22 s
epoch 10  loss 1.0798  time 24 s
epoch 11  loss 1.2188  time 27 s
epoch 12  loss 1.2717  time 30 s
epoch 13  loss 1.2410  time 33 s
epoch 14  loss 1.2204  time 35 s
epoch 15  loss 1.2015  time 38 s
epoch 16  loss 1.1524  time 40 s
epoch 17  loss 1.1397  time 43 s
epoch 18  loss 1.1343  time 46 s
epoch 19  loss 1.1298  time 48 s
epoch 20  loss 1.1272  time 51 s
epoch 21  loss 1.1280  time 54 s
epoch 22  loss 1.1260  time 57 s
epoch 23  loss 1.1235  time 59 s
epoch 24  loss 1.1229  time 62 s
epoch  1  loss 0.5640  time 0 s
epoch  2  loss 0.4954  time 3 s
epoch  3  loss 0.5022  time 6 s
epoch  4  loss 0.5396  time 8 s
epoch  5  loss 0.5988  time 11 s
epoch  6  loss 0

In [14]:
crown_linf_row = run_two_seeds("crown_on_linf", lambda m, x, y, t: my_loss(m, x, y, t, crown_norm="linf"))
results_c.append(crown_linf_row)
print_results(results_c)

epoch  1  loss 0.5883  time 0 s
epoch  2  loss 0.4640  time 4 s
epoch  3  loss 0.4887  time 7 s
epoch  4  loss 0.5152  time 10 s
epoch  5  loss 0.5715  time 13 s
epoch  6  loss 0.6481  time 17 s
epoch  7  loss 0.7459  time 20 s
epoch  8  loss 0.8623  time 23 s
epoch  9  loss 0.9951  time 26 s
epoch 10  loss 1.1387  time 30 s
epoch 11  loss 1.2838  time 33 s
epoch 12  loss 1.3435  time 36 s
epoch 13  loss 1.3223  time 39 s
epoch 14  loss 1.3097  time 43 s
epoch 15  loss 1.3001  time 46 s
epoch 16  loss 1.2624  time 49 s
epoch 17  loss 1.2532  time 52 s
epoch 18  loss 1.2498  time 56 s
epoch 19  loss 1.2469  time 59 s
epoch 20  loss 1.2449  time 62 s
epoch  1  loss 0.5640  time 0 s
epoch  2  loss 0.4687  time 4 s
epoch  3  loss 0.4962  time 7 s
epoch  4  loss 0.5351  time 10 s
epoch  5  loss 0.5969  time 13 s
epoch  6  loss 0.6899  time 16 s
epoch  7  loss 0.7878  time 20 s
epoch  8  loss 0.9058  time 23 s
epoch  9  loss 1.0323  time 26 s
epoch 10  loss 1.1630  time 29 s
epoch 11  loss 1

In [15]:
def l2_share(model, x, y, crown_norm=None):
    ce = {}
    for p in NORMS:
        center, tau = propagation_region(model, x, y, EPS[p], LAM[p], p)
        with torch.no_grad():
            m = ibp_margin_bounds(model, center, tau, p, y)
            if p == crown_norm:
                m = crown_ibp_margin_bounds(model, center, tau, p, y)
            ce[p] = F.cross_entropy(-m, y, reduction="none")
    return 100 * (ce["l2"] > ce["linf"]).float().mean().item()

xb, yb = x_test[:1000], y_test[:1000]
print(f"CURE-Max: l2 loss is the max on {l2_share(models['cure_max'], xb, yb):5.1f}% of images")
print(f"CROWN on l2: l2 loss is the max on {l2_share(models['cure_max_crown_l2'], xb, yb, 'l2'):5.1f}% of images")
print(f"CROWN on linf: l2 loss is the max on {l2_share(models['crown_on_linf'], xb, yb, 'linf'):5.1f}% of images")

CURE-Max: l2 loss is the max on  50.1% of images
CROWN on l2: l2 loss is the max on  46.5% of images
CROWN on linf: l2 loss is the max on  69.0% of images


In [16]:
for r in results_c:
    if r["name"] in ("cure_max", "cure_max_crown_l2", "crown_on_linf"):
        print(f"{r['name']:<26} gap linf {r['pgd_linf'] - r['linf']:5.2f}   gap l2 {r['pgd_l2'] - r['l2']:5.2f}")

cure_max                   gap linf  9.86   gap l2 21.78
cure_max_crown_l2          gap linf  5.22   gap l2 19.90
crown_on_linf              gap linf  9.61   gap l2 17.08
